In [1]:
# Import libraries
from pathlib import Path
import torch
import torch.nn as nn

from src.data.data_loader import create_dataloaders
from src.model.transformer import build_transformer
from src.model.transformer import Transformer
from src.train.training import train_model
from src.utils.utils import get_device
from nltk.tokenize import word_tokenize

from src.utils.constants import PADDING_ID, UNKNOWN_ID, START_OF_SENTENCE_ID, END_OF_SENTENCE_ID
from src.utils.constants import PADDING_VALUE, UNKNOWN_VALUE, START_OF_SENTENCE_VALUE, END_OF_SENTENCE_VALUE

/opt/anaconda3/lib/python3.11/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'dlopen(/opt/anaconda3/lib/python3.11/site-packages/torchvision/image.so, 0x0006): Symbol not found: __ZN3c1017RegisterOperatorsD1Ev
  Referenced from: <CFED5F8E-EC3F-36FD-AAA3-2C6C7F8D3DD9> /opt/anaconda3/lib/python3.11/site-packages/torchvision/image.so
  Expected in:     <CDAC6E34-8608-3E70-8B2F-32BCD38E90FB> /opt/anaconda3/lib/python3.11/site-packages/torch/lib/libtorch_cpu.dylib'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [2]:
# Initialize model and training parameters

# Size of embedding vector
d_model = 512
# Max sequence length for input words/tokens
seq_len = 100
# Dropout rate
dropout = 0.1
# number of encoder blocks
num_layers = 1
# number of attention heads
num_heads = 8
# Number of hidden nodes for feed-forward layer
d_ff = 4*d_model

# Number of epochs
epochs = 5
# Batch size for training
batch_size = 128

# Train file
train_file = './data/train/poems.txt'

In [3]:
# Get a device to use for training/inference
device = get_device()

# Create training and testing data loaders
train_dataloader, vocab = create_dataloaders(batch_size, seq_len, train_file)

print(f'Training data size: {len(train_dataloader) * batch_size}')

Number of tokenized words:  194655
Number of tokenized words after adding <eos>:  194755
Training data size: 194688


In [4]:
# Create encoder only transformer model
encoder_only_transformer_model = build_transformer(d_model, len(vocab), seq_len, dropout,
                                                   num_layers, num_heads, d_ff).to(device)

print(encoder_only_transformer_model)

Transformer(
  (embed): InputEmbedding(
    (embedding): Embedding(6993, 512)
  )
  (pos): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): Encoder(
    (layers): ModuleList(
      (0): EncoderBlock(
        (self_attention): MultiHeadAttention(
          (dropout): Dropout(p=0.1, inplace=False)
          (query_linear_layer): Linear(in_features=512, out_features=512, bias=True)
          (key_linear_layer): Linear(in_features=512, out_features=512, bias=True)
          (value_linear_layer): Linear(in_features=512, out_features=512, bias=True)
          (output_linear_layer): Linear(in_features=512, out_features=512, bias=True)
        )
        (feed_forward): FeedForward(
          (linear_1): Linear(in_features=512, out_features=2048, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear_2): Linear(in_features=2048, out_features=512, bias=True)
        )
        (dropout): Dropout(p=0.1, inplace=False)
        (norm): LayerN

In [5]:
# Train model

# Create optimizer and loss function
optimizer = torch.optim.Adam(encoder_only_transformer_model.parameters())
loss_fn = nn.CrossEntropyLoss()

# Start training the model
train_model(epochs, encoder_only_transformer_model, train_dataloader,
            loss_fn, optimizer, device)

Epoch 1, Train Loss 0.8359161615371704
Epoch 2, Train Loss 0.23569808900356293
Epoch 3, Train Loss 0.21092145144939423
Epoch 4, Train Loss 0.2001051902770996
Epoch 5, Train Loss 0.19378727674484253


In [6]:
# Save model

# Create models directory
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

# Create model save path
MODEL_NAME = "07_text_generation.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

In [7]:
# Save the model state dict
print(f"Saving model to: {MODEL_SAVE_PATH}")
torch.save(obj=encoder_only_transformer_model.state_dict(), f=MODEL_SAVE_PATH)

Saving model to: models/07_text_generation.pth
